# Token Bucket Rate Limiter (thread-safe, monotonic)

In [32]:
import time
import threading

In [74]:
class tokenBucket:
  def __init__(self, rate_per_sec: float, capacity: float):
    self.rate = rate_per_sec # tokens refil rate per second
    self.cap = capacity # maximum tokens a bucket can fill
    self.tokens = capacity # start accepting the requests with the burst capacity
    self.last = time.monotonic() # time passed since the last request. monotonic time because we don't want the wall cock jump
    self._lock = threading.Lock() # lock guard. in a multi-threaded server, multiple requests might hit the bucket at once. the lock guard prevents race condition where two requests share the last token.

  def allow(self, cost: float = 1.0) -> bool: # the logic
    with self._lock:
      now = time.monotonic()
      elapsed = now - self.last # calculate the time passed from last request
      
      if elapsed > 0:
        self.tokens = min(self.cap, self.tokens + elapsed * self.rate) # making sure that we are not crossing the maximum bucket capacity
        self.last = now # update the last request time

      if self.tokens >= cost: # check if we have enough budget for this request
        self.tokens -= cost # spend the token when a request hits
        return True, self.tokens
      
      return False, self.tokens # raet limit exceeded

In [75]:
def testBucket():
    # rate=2 (1 token every 0.5s), capacity=5
    bucket = tokenBucket(rate_per_sec=2, capacity=5)
    
    print(f"--- Testing Initial Burst (Capacity: {bucket.cap}) ---")
    for i in range(10):
        result = bucket.allow(1.0)
        print(f"Request {i+1}: {'ALLOWED' if result else 'REJECTED'} (Tokens left: {bucket.tokens:.2f})")
        time.sleep(0.2)

    print("\n--- Waiting 1 second (Should refill 2 tokens) ---")
    time.sleep(1)
    
    for i in range(3):
        result = bucket.allow(1.0)
        print(f"Request {i+8}: {'ALLOWED' if result else 'REJECTED'} (Tokens left: {bucket.tokens:.2f})")

    print("\n--- Testing Sustained Rate (0.2s intervals) ---")
    for i in range(5):
        time.sleep(0.2)
        result = bucket.allow(1.0)
        print(f"T + {0.2*(i+1):.1f}s | Request {i+11}: {'ALLOWED' if result else 'REJECTED'}")

In [76]:
if __name__ == "__main__":
    testBucket()

--- Testing Initial Burst (Capacity: 5) ---
Request 1: ALLOWED (Tokens left: 4.00)
Request 2: ALLOWED (Tokens left: 3.41)
Request 3: ALLOWED (Tokens left: 2.81)
Request 4: ALLOWED (Tokens left: 2.22)
Request 5: ALLOWED (Tokens left: 1.62)
Request 6: ALLOWED (Tokens left: 1.03)
Request 7: ALLOWED (Tokens left: 0.41)
Request 8: ALLOWED (Tokens left: 0.81)
Request 9: ALLOWED (Tokens left: 0.22)
Request 10: ALLOWED (Tokens left: 0.62)

--- Waiting 1 second (Should refill 2 tokens) ---
Request 8: ALLOWED (Tokens left: 2.03)
Request 9: ALLOWED (Tokens left: 1.03)
Request 10: ALLOWED (Tokens left: 0.03)

--- Testing Sustained Rate (0.2s intervals) ---
T + 0.2s | Request 11: ALLOWED
T + 0.4s | Request 12: ALLOWED
T + 0.6s | Request 13: ALLOWED
T + 0.8s | Request 14: ALLOWED
T + 1.0s | Request 15: ALLOWED


### Dynamic Cost Traffic Shaper

In traffic engineering, specifically for video streaming or media workflows (like what you'd see at Amagi), not all packets are created equal. A "request" isn't always 1.0 tokens. We call this **Variable Cost Rate Limiting.**

If you are streaming video, a Keyframe (I-Frame) is massive and computationally expensive, while a Delta Frame (P-Frame) is small. You want to rate-limit based on the actual load or data size. Implementing a Traffic Shaper for different video qualities

In [93]:
# constants representing the weights different traffic types

VIDEO_4K = 25 # Heavy Load
VIDEO_1080 = 10 # Medium Load
VIDEO_720 = 5 # light Load

bucket = tokenBucket(rate_per_sec = 20, capacity = 50)

def process_stream(quality_type, tag):
    allowed, remaining = bucket.allow(cost = quality_type)
    if allowed:
        print(f"[OK]... Transcoding {tag} (Cost: {quality_type})")
        print(f"        Remaining cost: {remaining:.2f}")
        
    else:
        print(f"[DENIED]... Capacity exceeded for {tag}. Dropping frame/Lowering bitrate")
        print(f"            Remaining cost: {remaining:.2f}")
        
# Simulation
process_stream(VIDEO_4K, "VIDEO_4K")
process_stream(VIDEO_4K, "VIDEO_4K")
print(f"sleep 1.5 sec. refilling 30 tokens")
time.sleep(1.5)
process_stream(VIDEO_4K, "VIDEO_4K")

[OK]... Transcoding VIDEO_4K (Cost: 25)
        Remaining cost: 25.00
[OK]... Transcoding VIDEO_4K (Cost: 25)
        Remaining cost: 0.32
sleep 1.5 sec. refilling 30 tokens
[OK]... Transcoding VIDEO_4K (Cost: 25)
        Remaining cost: 5.32
